# Co-Training with Qwen3-VL-8B Pseudo-Labels

Full experiment run: **2 tasks × 3 modalities × 4 budgets × 3 seeds = 72 experiments**

| Modality | Model |
| --- | --- |
| text_only | BERTweet (`vinai/bertweet-base`) |
| image_only | CLIP ViT (`openai/clip-vit-base-patch32`) |
| text_image | BERTweet + CLIP ViT (late fusion) |

Results saved under `results/cotrain/lg-cotrain/qwen3-vl-8b/{run_id}/...`  
Completed experiments are automatically skipped on re-run.

**Prerequisite:** Run `03_zeroshot_qwen3-8B.ipynb` first, then generate pseudo-labels:
```bash
python scripts/create_pseudo_labels.py --model qwen3-vl-8b
```

In [11]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import time
import pandas as pd
from lg_cotrain.run_all import run_all_experiments, format_summary_table

In [12]:
# ---- Configuration ----
RUN_ID = "run-1"  # Change to run-2, run-3 etc. for different settings
NUM_GPUS = 2      # Parallel experiments across GPUs (1 = sequential)

PARAMS = dict(
    pseudo_label_source="qwen3-vl-8b",
    run_id=RUN_ID,
    # Paper defaults
    weight_gen_epochs=7,
    cotrain_epochs=10,
    finetune_max_epochs=100,
    finetune_patience=5,
    batch_size=32,
    lr=2e-5,
)

DATA_ROOT = os.path.abspath("../data")
RESULTS_ROOT = os.path.abspath("../results")

TASKS = ["informative", "humanitarian"]
MODALITIES = ["text_only", "image_only", "text_image"]
BUDGETS = [5, 10, 25, 50]
SEED_SETS = [1, 2, 3]

total = len(TASKS) * len(MODALITIES) * len(BUDGETS) * len(SEED_SETS)
print(f"Run ID:       {RUN_ID}")
print(f"GPUs:         {NUM_GPUS}")
print(f"Data root:    {DATA_ROOT}")
print(f"Results root: {RESULTS_ROOT}")
print(f"Output path:  results/cotrain/lg-cotrain/qwen3-vl-8b/{RUN_ID}/{{task}}/{{modality}}/...")
print(f"Total experiments: {total}")

Run ID:       run-1
GPUs:         2
Data root:    D:\Workspace\Cotrain_CrisisMMD\data
Results root: D:\Workspace\Cotrain_CrisisMMD\results
Output path:  results/cotrain/lg-cotrain/qwen3-vl-8b/run-1/{task}/{modality}/...
Total experiments: 72


In [13]:
import gc, torch

all_summaries = {}  # (task, modality) -> results list

def run_batch(task, modality):
    """Run all 12 experiments for one (task, modality) and print summary."""
    print(f"\n{'='*70}")
    print(f"  {task} / {modality}  \u2014  {len(BUDGETS)}x{len(SEED_SETS)} = {len(BUDGETS)*len(SEED_SETS)} experiments")
    print(f"{'='*70}")
    
    n_total = len(BUDGETS) * len(SEED_SETS)
    progress = {"done": 0, "start": time.time()}

    def on_done(task, modality, budget, seed_set, status):
        progress["done"] += 1
        elapsed = time.time() - progress["start"]
        n = progress["done"]
        avg = elapsed / n
        eta = avg * (n_total - n)
        print(f"  [{n}/{n_total}] budget={budget}, seed={seed_set} -- {status} "
              f"({elapsed/60:.1f}min elapsed, ~{eta/60:.1f}min remaining)")

    start = time.time()
    results = run_all_experiments(
        task, modality,
        budgets=BUDGETS,
        seed_sets=SEED_SETS,
        num_gpus=NUM_GPUS,
        data_root=DATA_ROOT,
        results_root=RESULTS_ROOT,
        _on_experiment_done=on_done,
        **PARAMS,
    )
    elapsed = time.time() - start
    
    all_summaries[(task, modality)] = results
    
    print(f"\nCompleted {task}/{modality} in {elapsed/60:.1f}min")
    print(format_summary_table(results, task, modality,
                               budgets=BUDGETS, seed_sets=SEED_SETS))
    
    # Free GPU memory before next batch
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    return results

## Informative Task

In [14]:
run_batch("informative", "text_only")


  informative / text_only  —  4x3 = 12 experiments
Running 12 experiments in parallel across 2 GPUs...
  [1/12] budget=5, seed=2 -- done (23.4min elapsed, ~257.5min remaining)
  [2/12] budget=5, seed=1 -- done (26.9min elapsed, ~134.3min remaining)
  [3/12] budget=5, seed=3 -- done (46.0min elapsed, ~137.9min remaining)
  [4/12] budget=10, seed=1 -- done (54.1min elapsed, ~108.2min remaining)
  [5/12] budget=10, seed=2 -- done (69.4min elapsed, ~97.1min remaining)
  [6/12] budget=10, seed=3 -- done (81.5min elapsed, ~81.5min remaining)
  [7/12] budget=25, seed=1 -- done (92.6min elapsed, ~66.1min remaining)
  [8/12] budget=25, seed=2 -- done (108.1min elapsed, ~54.0min remaining)
  [9/12] budget=25, seed=3 -- done (116.9min elapsed, ~39.0min remaining)
  [10/12] budget=50, seed=1 -- done (133.6min elapsed, ~26.7min remaining)
  [11/12] budget=50, seed=2 -- done (139.5min elapsed, ~12.7min remaining)
  [12/12] budget=50, seed=3 -- done (159.5min elapsed, ~0.0min remaining)

Batch compl

[{'task': 'informative',
  'modality': 'text_only',
  'budget': 5,
  'seed_set': 1,
  'test_error_rate': np.float64(18.05736636245111),
  'test_macro_f1': 0.7549191452006504,
  'test_weighted_f1': 0.7980332272311016,
  'test_macro_precision': 0.8793562906223205,
  'test_macro_recall': 0.7292514254892896,
  'test_weighted_precision': 0.8492769433572123,
  'test_weighted_recall': 0.8194263363754889,
  'test_ece': 0.15228378038362853,
  'test_per_class_f1': [0.8806548901335631, 0.6291834002677377],
  'dev_error_rate': np.float64(19.96185632549269),
  'dev_macro_f1': 0.7241275692582663,
  'dev_weighted_f1': 0.7738262809576483,
  'dev_ece': 0.1749904563452738,
  'stopping_strategy': 'baseline',
  'phase1_seed_strategy': 'last',
  'phase1_best_epoch': None,
  'lambda1_mean': 1.0751206166927374,
  'lambda1_std': 0.10605982889159679,
  'lambda2_mean': 0.7843796181895544,
  'lambda2_std': 0.1061443276200764},
 {'task': 'informative',
  'modality': 'text_only',
  'budget': 5,
  'seed_set': 2,
  

In [ ]:
run_batch("informative", "image_only")


  informative / image_only  —  4x3 = 12 experiments
Running 12 experiments in parallel across 2 GPUs...
  [1/12] budget=5, seed=1 -- done (88.0min elapsed, ~968.5min remaining)
  [2/12] budget=5, seed=2 -- done (91.4min elapsed, ~456.9min remaining)
  [3/12] budget=10, seed=1 -- done (175.0min elapsed, ~525.1min remaining)
  [4/12] budget=5, seed=3 -- done (175.1min elapsed, ~350.2min remaining)
  [5/12] budget=10, seed=2 -- done (259.1min elapsed, ~362.8min remaining)
  [6/12] budget=10, seed=3 -- done (260.4min elapsed, ~260.4min remaining)


In [ ]:
run_batch("informative", "text_image")

## Humanitarian Task

In [6]:
run_batch("humanitarian", "text_only")


  humanitarian / text_only  —  4x3 = 12 experiments
Running 12 experiments in parallel across 2 GPUs...
  [1/12] budget=5, seed=2 -- done (13.4min elapsed, ~147.7min remaining)
  [2/12] budget=5, seed=1 -- done (15.4min elapsed, ~76.9min remaining)
  [3/12] budget=5, seed=3 -- done (26.9min elapsed, ~80.7min remaining)
  [4/12] budget=10, seed=1 -- done (30.4min elapsed, ~60.8min remaining)
  [5/12] budget=10, seed=2 -- done (40.2min elapsed, ~56.3min remaining)
  [6/12] budget=10, seed=3 -- done (45.5min elapsed, ~45.5min remaining)
  [7/12] budget=25, seed=1 -- done (53.3min elapsed, ~38.1min remaining)
  [8/12] budget=25, seed=2 -- done (61.0min elapsed, ~30.5min remaining)
  [9/12] budget=25, seed=3 -- done (66.7min elapsed, ~22.2min remaining)
  [10/12] budget=50, seed=1 -- done (75.8min elapsed, ~15.2min remaining)
  [11/12] budget=50, seed=2 -- done (79.9min elapsed, ~7.3min remaining)
  [12/12] budget=50, seed=3 -- done (90.7min elapsed, ~0.0min remaining)

Batch complete: 12 

[{'task': 'humanitarian',
  'modality': 'text_only',
  'budget': 5,
  'seed_set': 1,
  'test_error_rate': np.float64(27.95811518324607),
  'test_macro_f1': 0.6317537496297534,
  'test_weighted_f1': 0.7300996562513435,
  'test_macro_precision': 0.6199931288243443,
  'test_macro_recall': 0.7460803407257308,
  'test_weighted_precision': 0.7911879762353929,
  'test_weighted_recall': 0.7204188481675393,
  'test_ece': 0.22081082837743904,
  'test_per_class_f1': [0.25,
   0.7210884353741497,
   0.7476415094339622,
   0.7186311787072244,
   0.7214076246334311],
  'dev_error_rate': np.float64(27.15430861723447),
  'dev_macro_f1': 0.64175159108165,
  'dev_weighted_f1': 0.7404094747567411,
  'dev_ece': 0.22173308355655363,
  'stopping_strategy': 'baseline',
  'phase1_seed_strategy': 'last',
  'phase1_best_epoch': None,
  'lambda1_mean': 1.064351568289575,
  'lambda1_std': 0.19941869062434048,
  'lambda2_mean': 0.580764445855984,
  'lambda2_std': 0.15970086397396735},
 {'task': 'humanitarian',
  '

In [7]:
run_batch("humanitarian", "image_only")


  humanitarian / image_only  —  4x3 = 12 experiments
Running 12 experiments in parallel across 2 GPUs...
  [1/12] budget=5, seed=2 -- done (55.5min elapsed, ~610.2min remaining)
  [2/12] budget=5, seed=1 -- done (56.1min elapsed, ~280.4min remaining)
  [3/12] budget=5, seed=3 -- done (110.1min elapsed, ~330.3min remaining)
  [4/12] budget=10, seed=1 -- done (111.3min elapsed, ~222.5min remaining)
  [5/12] budget=10, seed=2 -- done (164.6min elapsed, ~230.4min remaining)
  [6/12] budget=10, seed=3 -- done (166.5min elapsed, ~166.5min remaining)
  [7/12] budget=25, seed=1 -- done (218.5min elapsed, ~156.1min remaining)
  [8/12] budget=25, seed=2 -- done (220.6min elapsed, ~110.3min remaining)
  [9/12] budget=25, seed=3 -- done (272.6min elapsed, ~90.9min remaining)
  [10/12] budget=50, seed=1 -- done (275.2min elapsed, ~55.0min remaining)
  [11/12] budget=50, seed=2 -- done (327.4min elapsed, ~29.8min remaining)
  [12/12] budget=50, seed=3 -- done (328.6min elapsed, ~0.0min remaining)



[{'task': 'humanitarian',
  'modality': 'image_only',
  'budget': 5,
  'seed_set': 1,
  'test_error_rate': np.float64(20.418848167539274),
  'test_macro_f1': 0.6728574112441692,
  'test_weighted_f1': 0.7917605678534385,
  'test_macro_precision': 0.7086919153134501,
  'test_macro_recall': 0.6549487785657998,
  'test_weighted_precision': 0.8004453475949903,
  'test_weighted_recall': 0.7958115183246073,
  'test_ece': 0.1335878690187844,
  'test_per_class_f1': [0.3333333333333333,
   0.7862068965517242,
   0.8489634748272458,
   0.7862595419847328,
   0.6095238095238096],
  'dev_error_rate': np.float64(21.14228456913828),
  'dev_macro_f1': 0.6587057299502183,
  'dev_weighted_f1': 0.787754763929278,
  'dev_ece': 0.152473475268943,
  'stopping_strategy': 'baseline',
  'phase1_seed_strategy': 'last',
  'phase1_best_epoch': None,
  'lambda1_mean': 1.0862887630921152,
  'lambda1_std': 0.13962599204197143,
  'lambda2_mean': 0.645232944754574,
  'lambda2_std': 0.1629241374548309},
 {'task': 'huma

In [8]:
run_batch("humanitarian", "text_image")


  humanitarian / text_image  —  4x3 = 12 experiments
Running 12 experiments in parallel across 2 GPUs...
  [1/12] budget=5, seed=2 -- done (72.6min elapsed, ~798.4min remaining)
  [2/12] budget=5, seed=1 -- done (74.6min elapsed, ~373.1min remaining)
  [3/12] budget=5, seed=3 -- done (140.9min elapsed, ~422.6min remaining)
  [4/12] budget=10, seed=1 -- done (148.1min elapsed, ~296.1min remaining)
  [5/12] budget=10, seed=2 -- done (210.6min elapsed, ~294.8min remaining)
  [6/12] budget=10, seed=3 -- done (217.8min elapsed, ~217.8min remaining)
  [7/12] budget=25, seed=1 -- done (277.7min elapsed, ~198.4min remaining)
  [8/12] budget=25, seed=2 -- done (287.2min elapsed, ~143.6min remaining)
  [9/12] budget=25, seed=3 -- done (344.2min elapsed, ~114.7min remaining)
  [10/12] budget=50, seed=1 -- done (359.4min elapsed, ~71.9min remaining)
  [11/12] budget=50, seed=2 -- done (409.7min elapsed, ~37.2min remaining)
  [12/12] budget=50, seed=3 -- done (428.5min elapsed, ~0.0min remaining)


[{'task': 'humanitarian',
  'modality': 'text_image',
  'budget': 5,
  'seed_set': 1,
  'test_error_rate': np.float64(19.999999999999996),
  'test_macro_f1': 0.7118449674148792,
  'test_weighted_f1': 0.8072440233868626,
  'test_macro_precision': 0.7154371135677218,
  'test_macro_recall': 0.8075901534766782,
  'test_weighted_precision': 0.8424547106432081,
  'test_weighted_recall': 0.8,
  'test_ece': 0.09838726208472132,
  'test_per_class_f1': [0.3902439024390244,
   0.7424242424242424,
   0.8418756815703381,
   0.7641681901279708,
   0.8205128205128205],
  'dev_error_rate': np.float64(20.140280561122246),
  'dev_macro_f1': 0.6706159463966642,
  'dev_weighted_f1': 0.8032323442615185,
  'dev_ece': 0.08696562374283177,
  'stopping_strategy': 'baseline',
  'phase1_seed_strategy': 'last',
  'phase1_best_epoch': None,
  'lambda1_mean': 1.0886902255267605,
  'lambda1_std': 0.13181735002854086,
  'lambda2_mean': 0.6537609882503069,
  'lambda2_std': 0.16820797787787659},
 {'task': 'humanitarian

## Cross-Modality Summary

In [9]:
import numpy as np

rows = []
for (task, modality), results in all_summaries.items():
    valid = [r for r in results if r is not None]
    if not valid:
        rows.append({"Task": task, "Modality": modality,
                     "N": 0, "Mean F1": "-", "Std F1": "-",
                     "Mean Err%": "-", "Mean ECE": "-"})
        continue
    f1s = [r["test_macro_f1"] for r in valid]
    errs = [r["test_error_rate"] for r in valid]
    eces = [r["test_ece"] for r in valid]
    rows.append({
        "Task": task,
        "Modality": modality,
        "N": len(valid),
        "Mean F1": f"{np.mean(f1s):.4f}",
        "Std F1": f"{np.std(f1s):.4f}",
        "Mean Err%": f"{np.mean(errs):.2f}",
        "Mean ECE": f"{np.mean(eces):.4f}",
    })

df = pd.DataFrame(rows)
print(f"Cross-Modality Summary ({RUN_ID})")
print(f"Averaged over {len(BUDGETS)} budgets x {len(SEED_SETS)} seeds = {len(BUDGETS)*len(SEED_SETS)} experiments each")
print("=" * 70)
df

Cross-Modality Summary (run-1)
Averaged over 4 budgets x 3 seeds = 12 experiments each


,Task,Modality,N,Mean F1,Std F1,Mean Err%,Mean ECE
0,informative,image_only,12,0.8488,0.0068,13.22,0.0890
1,informative,text_image,12,0.8506,0.0225,12.28,0.0739
2,humanitarian,text_only,12,0.6348,0.0087,26.31,0.1959
3,humanitarian,image_only,12,0.7131,0.0314,19.69,0.1131
4,humanitarian,text_image,12,0.7252,0.0177,18.66,0.0777


## Cleanup

In [10]:
import torch
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    print(f"GPU memory freed. Peak: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
else:
    print("No GPU available (ran on CPU)")

n_success = sum(1 for results in all_summaries.values() for r in results if r is not None)
print(f"\nDone. {n_success}/{total} experiments completed successfully.")

GPU memory freed. Peak: 0.00 GB

Done. 60/72 experiments completed successfully.
